In [1]:
from pylib.setup import *
setup_notebook()

!pip install comext_wrapper --upgrade --quiet

from comext_wrapper import ServicesApi
api = ServicesApi()

# Services trade API — lookup notebook

Use this notebook to explore the Eurostat BOP_ITS6_DET dataset (EU international trade in services, EBOPS 2010) before building queries.

Three tools:
- `api.info()` — dataset overview and dimension summary
- `api.codes(dimension, search=...)` — browse valid codes
- `api.get_data(...)` — fetch data as a DataFrame

---
## 1. Dataset overview

In [2]:
api.info() ;

Dataset : BOP_ITS6_DET
Name    : International trade in services (since 2010) (BPM6)
Updated : 2014-12-18T15:59:07+0100

dimension  # codes first code            first label      last code                               last label
     freq       12          P           Pluri-annual             A3                                Triannual
 currency      194      TOTAL Total (all currencies)            UNK                                  Unknown
 bop_item      383         CA        Current account FA__P__F5__JPY Financial account; portfolio investment;
 stk_flow      194      TOTAL                  Total            UNK                                  Unknown
  partner      488        EUR                 Europe            UNK                                  Unknown
      geo     4292        EUR                 Europe            UNK                                  Unknown

Key order: freq.currency.bop_item.stk_flow.partner.geo


---
## 2. Browse dimension codes

Valid dimensions: `freq`, `currency`, `bop_item`, `stk_flow`, `partner`, `geo`

In [3]:
# Flow and frequency options
print("--- stk_flow ---")
display(api.codes("stk_flow"))

print("--- freq ---")
display(api.codes("freq"))

--- stk_flow ---


,id,label
0,TOTAL,Total
1,CRE_DEB_AVG,Average of credits and debits
2,CRE_DEB_SUM,Sum of credits and debits
3,CRE,Credit
4,DEB,Debit
...,...,...
189,N_KA,Net other changes in the volume of assets/liab...
190,NE_LE,Net liabilities (liabilities minus assets)
191,HUMD_PC,Humidity (%)
192,NSP,Not specified


--- freq ---


,id,label
0,P,Pluri-annual
1,A,Annual
2,S,"Half-yearly, semesterly"
3,Q,Quarterly
4,M,Monthly
5,W,Weekly
6,B,Daily - business week
7,D,Daily
8,H,Hourly
9,I,Irregular / A-periodic


In [4]:
# Available currencies
api.codes("currency")

,id,label
0,TOTAL,Total (all currencies)
1,EUR,Euro
2,EUR_FOR,"Euro, if Euro is not the national currency"
3,MIO_EUR,Million euro
4,BGN,Bulgarian lev
...,...,...
189,NAT,National currency (former currencies of the eu...
190,FOR,Foreign currency
191,OTH,Other currencies
192,NAP,Not applicable


In [5]:
# Partner aggregates
api.codes("partner", aggregates_only=True)

,id,label
0,EUR,Europe
1,EU27_2020,European Union - 27 countries (from 2020)
2,EU28,European Union - 28 countries (2013-2020)
3,EU27_2007,European Union - 27 countries (2007-2013)
4,EU25,European Union - 25 countries (2004-2006)
...,...,...
233,NAL,Not allocated
234,NAP,Not applicable
235,NRP,No response
236,NSP,Not specified


In [6]:
# Find reporter codes by country name
api.codes("geo", search="denmark")

,id,label
0,EU27_2020_X_DK,European Union - 27 countries (from 2020) exce...
1,DK,Denmark
2,EU27_2020_NEA21_X_DK_SE,EU27 countries (from 2020) not in EA21 (from 2...
3,EU27_2020_NEA20_X_DK_SE,EU27 countries (from 2020) not in EA20 (2023-2...


In [7]:
# Browse EBOPS service categories
api.codes("bop_item", search="services")

,id,label
0,GS,Goods and services
1,GSIN1,"Goods, services and primary income"
2,S,Services
3,ISS,International supply of services (ISS)
4,SA,Services: manufacturing services on physical i...
...,...,...
147,SL33X,"Memo item: services, government services: othe..."
148,SOX,Services: commercial services
149,SN,Services: services not allocated
150,S_X_CG,Services excluding transport and financial ser...


In [8]:
# Search for a specific service type
api.codes("bop_item", search="transport")

,id,label
0,SC,Services: transport
1,SC1,Services: sea transport
2,SC11,Services: sea transport; passenger
3,SC12,Services: sea transport; freight
4,SC13,Services: sea transport; other than passenger ...
5,SC2,Services: air transport
6,SC21,Services: air transport; passenger
7,SC22,Services: air transport; freight
8,SC23,Services: air transport; other than passenger ...
9,SC3,Services: other modes of transport


---
## 3. Fetch data

Use `+` to combine multiple values per dimension.

In [9]:
df = api.get_data(
    geo          = "DK+DE+FR",
    partner      = "WRL_REST",   # rest of world — check api.codes('partner', aggregates_only=True)
    bop_item     = "S",          # total services
    stk_flow     = "CRE+DEB",   # credit (exports) + debit (imports)
    currency     = "MIO_EUR",
    start_period = "2015",
    end_period   = "2023",
)

df

,freq,currency,bop_item,stk_flow,partner,geo,period,value
0,A,MIO_EUR,S,CRE,WRL_REST,DK,2015,58384.0
1,A,MIO_EUR,S,CRE,WRL_REST,DK,2016,57751.2
2,A,MIO_EUR,S,CRE,WRL_REST,DK,2017,64111.0
3,A,MIO_EUR,S,CRE,WRL_REST,DK,2018,70279.1
4,A,MIO_EUR,S,CRE,WRL_REST,DK,2019,74704.6
5,A,MIO_EUR,S,CRE,WRL_REST,DK,2020,67072.3
6,A,MIO_EUR,S,CRE,WRL_REST,DK,2021,84870.4
7,A,MIO_EUR,S,CRE,WRL_REST,DK,2022,129014.5
8,A,MIO_EUR,S,CRE,WRL_REST,DK,2023,111623.4
9,A,MIO_EUR,S,CRE,WRL_REST,DE,2015,258520.0


In [10]:
# Quick pivot: period x geo, exports only
exports = df[df["stk_flow"] == "CRE"].pivot_table(
    index="period", columns="geo", values="value"
)
exports

geo,DE,DK,FR
period,,,
2015,258520.0,58384.0,230471.0
2016,270545.0,57751.2,234148.0
2017,290027.0,64111.0,242931.0
2018,308653.0,70279.1,256348.0
2019,327246.0,74704.6,264183.0
2020,292006.0,67072.3,215078.0
2021,348863.0,84870.4,267513.0
2022,419691.0,129014.5,345712.0
2023,418631.0,111623.4,346396.0


---
## 4. Multi-item query from a list

Use `"+".join(...)` to build the key string from a Python list.

In [11]:
# Sub-categories of services — check api.codes('bop_item') for full list
items = ["SC", "SD", "SE"]   # transport, travel, other services
geos  = ["DK", "DE", "FR", "SE"]

item_key = "+".join(items)
geo_key  = "+".join(geos)

df2 = api.get_data(
    geo          = geo_key,
    partner      = "WRL_REST",
    bop_item     = item_key,
    stk_flow     = "CRE+DEB",
    currency     = "MIO_EUR",
    start_period = "2015",
    end_period   = "2023",
)

df2

,freq,currency,bop_item,stk_flow,partner,geo,period,value
0,A,MIO_EUR,SC,CRE,WRL_REST,DK,2015,33054.2
1,A,MIO_EUR,SC,CRE,WRL_REST,DK,2016,29504.5
2,A,MIO_EUR,SC,CRE,WRL_REST,DK,2017,33222.4
3,A,MIO_EUR,SC,CRE,WRL_REST,DK,2018,35837.5
4,A,MIO_EUR,SC,CRE,WRL_REST,DK,2019,39682.0
...,...,...,...,...,...,...,...,...
211,A,MIO_EUR,SE,DEB,WRL_REST,SE,2019,1771.9
212,A,MIO_EUR,SE,DEB,WRL_REST,SE,2020,1268.4
213,A,MIO_EUR,SE,DEB,WRL_REST,SE,2021,1459.6
214,A,MIO_EUR,SE,DEB,WRL_REST,SE,2022,1731.5
